In [10]:
!pip install groq

In [11]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('EGROQ_API_KEY')
client = Groq(api_key=api_key)
print("Connected!")

Connected!


In [12]:
sample_email_subject="URGENT: Your account has been locked.Verify Now!"
sample_email_body="Dear customer,we have detected unusual activity on your account.Please verify your identity immediately by clicking the link below.Failure to do so within 24 hours may result in permanent aaccount suspension."

In [16]:
def triage_agent(subject,body):
  prompt=f"""
  You are an email security triage agent.Analyze the following email and classify it as one of:Important,Spam, or Suspicious.

  Look for these patterns:
  -Urgency-based language(e.g,"act now", "24 hours","immediately")
  -Requests for sensitive information or account verification
  -Threats of account suspension or loss
  -Generic greetings instead of personalized ones.


Subject:{subject}
Body:{body}

Respond ONLY in this JSON format:
{{
"classification":"Important,Spam,or Suspicious",
"reasons":["reason1","reason2"],
"recommendation":"what the user should do"
}}
"""
  response=client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":prompt}]
  )
  return response.choices[0].message.content

#Test
result=triage_agent(sample_email_subject,sample_email_body)
print(result)
#

{
  "classification": "Suspicious",
  "reasons": ["Urgency-based language", "Requests for verification by clicking a link", "Threats of account suspension"],
  "recommendation": "Do not click on the link, but instead log in directly to your account or contact the company's support to verify the authenticity of the email."
}


## TESTING IMPORTANT EMAIL AND SPAM MAIL

In [17]:
# Important email test
important_subject = "Meeting rescheduled to 3 PM tomorrow"
important_body = "Hi team, our project sync meeting has been moved from 10 AM to 3 PM tomorrow due to a scheduling conflict. Please update your calendars. Thanks, Sarah"

result_important = triage_agent(important_subject, important_body)
print("IMPORTANT TEST:")
print(result_important)
print()

IMPORTANT TEST:
{
"classification":"Important",
"reasons":["Urgency-based language is related to meeting rescheduling, which was explicitly stated but not using alarming language, yet it was clear what the urgency was. Generic greeting is 'team', which is acceptable for a company or work group communication."()],
"recommendation":"Update calendar with the new meeting time."
}



In [18]:
# Spam email test
spam_subject = "You've won a $1000 gift card!"
spam_body = "Congratulations! You have been randomly selected to receive a free $1000 Amazon gift card. Click here to claim your prize now before it expires!"

result_spam = triage_agent(spam_subject, spam_body)
print("SPAM TEST:")
print(result_spam)

SPAM TEST:
{
"classification":"Spam",
"reasons":["Urgency-based language (act now)","Suspicious tone (random prize award and expiration notice)","Request for immediate action without validation"],
"recommendation":"Be cautious and verify the authenticity of this email. Do not click on the link as it may be malicious."
}
